# J3-04 · MCP : un serveur d'outils local, et la sécurité de la chaîne d'approvisionnement

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 3 · créneau 13h20–14h35 · ⏱️ ~1h15 (dont 40 min de pratique)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Comprendre ce qu'apporte le **Model Context Protocol** · 2. Écrire un serveur MCP avec **FastMCP** et ses **annotations** · 3. Le consommer avec `langchain.mcp` (**bêta**, LangChain 1.4) en processus, en **stdio** ou en HTTP · 4. Appliquer une **liste blanche** côté client · 5. Raisonner « chaîne d'approvisionnement » (OWASP **ASI04**) |
| 🧱 **Fil rouge** | Le spécialiste Diagnostic accède à l'**inventaire du parc** via MCP |
| ✅ **Prérequis** | J3-02 (spécialistes) |
| 📦 **Versions** | `langchain 1.4` (`langchain[mcp]`) · `fastmcp 4.0` · `mcp 2.2` |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. MCP en une phrase

### [Théorie] La prise électrique standard des outils

*Analogie :* avant les prises normalisées, chaque appareil avait son câble. MCP standardise la **prise** entre un agent (le client) et des fournisseurs d'outils (les serveurs) : un serveur écrit une fois sert à LangChain, à un IDE, à un autre framework.

| Sans MCP | Avec MCP |
|---|---|
| chaque outil est une fonction Python dans **votre** code | les outils vivent dans un **serveur** séparé, découvert à l'exécution |
| ajouter un outil = modifier l'agent | ajouter un outil = le déclarer côté serveur |
| tout tourne dans un seul processus | serveur en processus, en sous-processus (stdio) ou à distance (HTTP) |

> ⚠️ **Le revers :** brancher un serveur MCP, c'est **exécuter le code de quelqu'un d'autre** et **donner des outils à son agent**. Un serveur inconnu est un risque de chaîne d'approvisionnement (OWASP **ASI04**), au même titre qu'une dépendance PyPI non vérifiée.

## 3. Le serveur : `helpdesk_ops/mcp_inventaire.py`

Quatre outils sur un inventaire **fictif** : trois en lecture, un destructif. Chaque outil déclare des **annotations** :

| Annotation | Sens |
|---|---|
| `readOnlyHint` | l'outil ne modifie rien |
| `destructiveHint` | l'outil peut détruire (supprimer, réinitialiser) |
| `idempotentHint` | le rappeler avec les mêmes arguments ne change rien de plus |

Ce sont des **déclarations du serveur**, pas des garanties : un serveur malveillant peut mentir.

In [ ]:
import inspect

import helpdesk_ops.mcp_inventaire as inventaire

source = inspect.getsource(inventaire)
debut, fin = source.index("@mcp.tool(annotations={\"readOnlyHint\": True, \"idempotentHint\": True})\ndef detail_poste"), source.index("def annotations")
print(source[debut:fin])

## 4. Consommer le serveur : `MCPAdapter`

### [Théorie] Trois modes de connexion, du plus simple au plus réaliste

| Cible passée à `MCPAdapter` | Transport | Usage |
|---|---|---|
| l'objet `FastMCP` | en processus | notebooks, tests |
| un `Path` vers le script, ou un transport stdio explicite | **stdio** : le serveur est lancé en sous-processus | outil local, isolé dans son processus |
| une URL `http(s)://…` | HTTP | serveur distant (ou local sur `127.0.0.1`) |

`MCPAdapter` **refuse** une simple chaîne qui n'est pas une URL : `fastmcp` l'interpréterait comme un fichier à **exécuter**. Pour lancer un script local, il faut le dire explicitement avec `Path`.

In [ ]:
from pathlib import Path

from langchain.mcp import MCPAdapter

outils = await MCPAdapter(inventaire.mcp).list_tools()   # en processus
for o in outils:
    print(f"{o.name:<26} annotations={inventaire.annotations(o)}")

print("\nAppel direct :", await outils[[o.name for o in outils].index("detail_poste")].ainvoke({"poste": "PC-0203"}))

In [ ]:
# 1. Une chaîne qui n'est pas une URL est refusée (elle pourrait désigner un script à exécuter)
try:
    await MCPAdapter("mcp_inventaire.py").list_tools()
except ValueError as erreur:
    print("❌", str(erreur)[:150], "…")

# 2. stdio : le serveur tourne dans un SOUS-PROCESSUS (même interpréteur que le notebook).
#    Dans un script, MCPAdapter(Path(inventaire.__file__)) suffit. Dans Jupyter sous Windows, le stderr du noyau
#    n'a pas de descripteur de fichier : on redirige celui du serveur vers un journal (transport_stdio).
outils_stdio = await MCPAdapter(inventaire.transport_stdio()).list_tools()
print("✅ stdio :", [o.name for o in outils_stdio])
print("   journal du serveur :", inventaire.transport_stdio().log_file)

**👀 Ce que vous devez observer :** la chaîne est refusée avec un message qui explique pourquoi ; en stdio, les mêmes quatre outils sont découverts, mais le serveur tourne dans un processus séparé (il pourrait planter sans emporter le notebook), et son journal est écrit dans `data/logs/` (ignoré par Git).

> ⚠️ **Vu en préparant ce notebook :** `MCPAdapter(Path(...))`, qui fonctionne dans un script, échoue dans Jupyter sous Windows avec « Client failed to connect: fileno ». Le client transmet le `stderr` du noyau au sous-processus, et celui-ci n'a pas de vrai descripteur de fichier. D'où `log_file`.

> 💡 **Le « Tips » du Formateur : 100 % local, vraiment ?** Au démarrage d'un serveur, FastMCP **vérifie les mises à jour sur PyPI** (constaté en préparant le cours : bannière « Update available »). `generate_env.py` pose désormais `FASTMCP_CHECK_FOR_UPDATES=off` et `FASTMCP_SHOW_SERVER_BANNER=false`. Une bibliothèque qui « appelle la maison » est un flux sortant à inventorier.

## 5. La liste blanche côté client

### [Théorie] Ne jamais exposer à l'agent tout ce que le serveur propose

Le client décide **quels** outils il transmet à l'agent :
1. une **liste blanche par nom** (ce dont le spécialiste a besoin, rien de plus) ;
2. un **refus des outils destructifs**, et de ceux qui ne se déclarent pas en lecture seule ;
3. pour les rares outils d'action nécessaires : une **validation humaine** (J2).

In [ ]:
retenus = inventaire.filtrer_outils(outils)
print("refus des destructifs :", [o.name for o in retenus])

retenus = inventaire.filtrer_outils(outils, autorises={"detail_poste", "historique_interventions"})
print("liste blanche stricte :", [o.name for o in retenus])

**👀 Ce que vous devez observer :** `reinitialiser_poste` n'est jamais transmis ; avec la liste blanche, seuls les deux outils utiles au diagnostic restent (`lister_postes` n'est pas nécessaire au spécialiste).

## 6. Le spécialiste Diagnostic, augmenté par MCP

In [ ]:
import time

from helpdesk_ops.multi_agents import construire_specialistes

specialistes = construire_specialistes(llm, outils_mcp=retenus)
diagnostic = specialistes["diagnostic"]

debut = time.perf_counter()
with propagate_attributes(session_id="j3-04-mcp", tags=["jour-3", "mcp"]):
    etat = await diagnostic.ainvoke(
        {"messages": [("user", "Le poste PC-0203 est très lent : que sait-on de lui et de ses interventions passées ?")]},
        config={"callbacks": [langfuse_handler], "run_name": "diagnostic-mcp", "recursion_limit": 15})
print(f"⏱ {time.perf_counter() - debut:.0f} s")
print("outils appelés :", [c["name"] for m in etat["messages"] for c in (getattr(m, "tool_calls", None) or [])])
print("🤖", etat["messages"][-1].content)
langfuse.flush()

**👀 Ce que vous devez observer :** le spécialiste appelle `detail_poste` et/ou `historique_interventions` ; la réponse rapproche l'âge du poste (2019), son système (Windows 10) et ses lenteurs passées. Dans Langfuse, les appels MCP apparaissent comme des outils ordinaires : pour l'agent, un outil MCP est un outil comme un autre.

## 7. 🛡️ OWASP ASI04 : la check-list avant de brancher un serveur MCP

| Question | Pour notre serveur |
|---|---|
| **Provenance** : qui l'a écrit, où est le code source ? | écrit par nous, dans le dépôt, relu |
| **Versions figées** : dépendances et serveur épinglés ? | `fastmcp` figé par `uv.lock` |
| **Moindre privilège** : quels outils, quels droits ? | liste blanche, aucun outil destructif transmis |
| **Exposition réseau** : qui peut le joindre ? | en processus ou stdio ; en HTTP, `127.0.0.1` uniquement |
| **Flux sortants** : appelle-t-il l'extérieur ? | non, et la vérification de mise à jour est coupée |
| **Données** : que voit-il passer ? | des noms de postes et des identifiants pseudonymisés |
| **Traçabilité** : ses appels sont-ils journalisés ? | oui, dans les traces Langfuse du spécialiste |

> 🛡️ **Rappel :** les annotations sont des **promesses** du serveur. Un serveur tiers qui déclare `readOnlyHint` sur un outil qui efface reste un outil qui efface. La liste blanche **par nom**, après lecture du code, est la seule défense fiable.

## 🧭 Récap

| Vous savez maintenant… | API |
|---|---|
| Écrire un serveur MCP | `FastMCP("nom")`, `@mcp.tool(annotations={...})`, `mcp.run()` |
| Le consommer | `await MCPAdapter(serveur \| Path(...) \| "http://…").list_tools()` |
| Lire les annotations | `outil.metadata["mcp"]["tool"]["annotations"]` (`helpdesk_ops.mcp_inventaire.annotations`) |
| Filtrer | `filtrer_outils(outils, autorises={...})` |
| Sécuriser | check-list ASI04 ; `FASTMCP_CHECK_FOR_UPDATES=off` |

### ✅ Quiz express

1. Pourquoi `MCPAdapter("serveur.py")` est-il refusé ?
2. Une annotation `readOnlyHint=True` garantit-elle qu'un outil ne modifie rien ?
3. Quel avantage apporte le transport stdio par rapport au mode en processus ?
4. Citez trois questions de la check-list ASI04.

<details>
<summary>Réponses</summary>

1. `fastmcp` interpréterait la chaîne comme un fichier à exécuter : il faut demander explicitement un script local avec `Path`.
2. Non : c'est une déclaration du serveur ; seule la lecture du code et une liste blanche par nom font foi.
3. Le serveur tourne dans un processus séparé : isolation (un plantage ne fait pas tomber l'agent) et séparation des droits.
4. Provenance, versions figées, moindre privilège, exposition réseau, flux sortants, données, traçabilité.
</details>